# main abaltion

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import RobertaTokenizer, RobertaModel
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import Data, Batch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from tqdm import tqdm
import warnings
import ast
warnings.filterwarnings('ignore')

import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"


class ICFGBuilder:
    def __init__(self, graphcodebert_model):
        self.graphcodebert = graphcodebert_model
        
    def build_icfg(self, code_snippet):
        try:
            tree = ast.parse(code_snippet)
            nodes = []
            edges = []
            node_id = 0
            func_calls = {}
            func_defs = {}
            
            def traverse(node, parent_id=None):
                nonlocal node_id
                current_id = node_id
                node_type = type(node).__name__
                nodes.append(node_type)
                node_id += 1
                
                if parent_id is not None:
                    edges.append([parent_id, current_id])
                
                if isinstance(node, ast.FunctionDef):
                    func_defs[node.name] = current_id
                
                if isinstance(node, ast.Call):
                    if isinstance(node.func, ast.Name):
                        func_name = node.func.id
                        if func_name not in func_calls:
                            func_calls[func_name] = []
                        func_calls[func_name].append(current_id)
                
                for child in ast.iter_child_nodes(node):
                    traverse(child, current_id)
            
            traverse(tree)
            
            for func_name, call_sites in func_calls.items():
                if func_name in func_defs:
                    def_id = func_defs[func_name]
                    for call_id in call_sites:
                        edges.append([call_id, def_id])
                        edges.append([def_id, call_id])
            
            if len(nodes) == 0:
                nodes = ['Module']
                edges = []
            
            return nodes, edges
        except:
            return ['Module'], []
    
    def nodes_to_embeddings(self, nodes, tokenizer, device, frozen=False):
        embeddings = []
        for node_type in nodes:
            tokens = tokenizer(node_type, return_tensors='pt', padding=True, truncation=True, max_length=16)
            tokens = {k: v.to(device) for k, v in tokens.items()}
            if frozen:
                with torch.no_grad():
                    outputs = self.graphcodebert(**tokens)
                    embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            else:
                outputs = self.graphcodebert(**tokens)
                embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            embeddings.append(embedding)
        return torch.stack(embeddings)


class DFGBuilder:
    def __init__(self, graphcodebert_model):
        self.graphcodebert = graphcodebert_model
        
    def build_dfg(self, code_snippet):
        try:
            tree = ast.parse(code_snippet)
            nodes = []
            edges = []
            node_id = 0
            var_last_write = {}
            var_last_read = {}
            
            def extract_dataflow(node, current_id):
                if isinstance(node, ast.Name):
                    var_name = node.id
                    
                    if isinstance(node.ctx, ast.Store):
                        if var_name in var_last_read:
                            for read_id in var_last_read[var_name]:
                                edges.append([read_id, current_id])
                        var_last_write[var_name] = current_id
                        var_last_read[var_name] = []
                    
                    elif isinstance(node.ctx, ast.Load):
                        if var_name in var_last_write:
                            edges.append([var_last_write[var_name], current_id])
                        if var_name not in var_last_read:
                            var_last_read[var_name] = []
                        var_last_read[var_name].append(current_id)
                
                for child in ast.iter_child_nodes(node):
                    extract_dataflow(child, current_id)
            
            def traverse(node):
                nonlocal node_id
                current_id = node_id
                nodes.append(type(node).__name__)
                node_id += 1
                extract_dataflow(node, current_id)
                for child in ast.iter_child_nodes(node):
                    traverse(child)
            
            traverse(tree)
            
            if len(nodes) == 0:
                nodes = ['Program']
                edges = []
            
            return nodes, edges
        except:
            return ['Program'], []
    
    def nodes_to_embeddings(self, nodes, tokenizer, device, frozen=False):
        embeddings = []
        for node_type in nodes:
            tokens = tokenizer(node_type, return_tensors='pt', padding=True, truncation=True, max_length=16)
            tokens = {k: v.to(device) for k, v in tokens.items()}
            if frozen:
                with torch.no_grad():
                    outputs = self.graphcodebert(**tokens)
                    embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            else:
                outputs = self.graphcodebert(**tokens)
                embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            embeddings.append(embedding)
        return torch.stack(embeddings)


class CDGBuilder:
    def __init__(self, graphcodebert_model):
        self.graphcodebert = graphcodebert_model
        
    def build_cdg(self, code_snippet):
        try:
            tree = ast.parse(code_snippet)
            nodes = []
            edges = []
            node_id = 0
            control_stack = []
            
            def traverse(node, parent_id=None):
                nonlocal node_id
                current_id = node_id
                node_type = type(node).__name__
                nodes.append(node_type)
                node_id += 1
                
                if parent_id is not None:
                    edges.append([parent_id, current_id])
                
                if isinstance(node, (ast.If, ast.For, ast.While, ast.Try)):
                    for ctrl_id in control_stack:
                        edges.append([ctrl_id, current_id])
                    control_stack.append(current_id)
                    
                    for child in ast.iter_child_nodes(node):
                        traverse(child, current_id)
                    
                    control_stack.pop()
                else:
                    if control_stack:
                        for ctrl_id in control_stack:
                            edges.append([ctrl_id, current_id])
                    
                    for child in ast.iter_child_nodes(node):
                        traverse(child, current_id)
            
            traverse(tree)
            
            if len(nodes) == 0:
                nodes = ['Program']
                edges = []
            
            return nodes, edges
        except:
            return ['Program'], []
    
    def nodes_to_embeddings(self, nodes, tokenizer, device, frozen=False):
        embeddings = []
        for node_type in nodes:
            tokens = tokenizer(node_type, return_tensors='pt', padding=True, truncation=True, max_length=16)
            tokens = {k: v.to(device) for k, v in tokens.items()}
            if frozen:
                with torch.no_grad():
                    outputs = self.graphcodebert(**tokens)
                    embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            else:
                outputs = self.graphcodebert(**tokens)
                embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            embeddings.append(embedding)
        return torch.stack(embeddings)


class ASTBuilder:
    def __init__(self, graphcodebert_model):
        self.graphcodebert = graphcodebert_model
        
    def build_ast(self, code_snippet):
        try:
            tree = ast.parse(code_snippet)
            nodes = []
            edges = []
            node_id = 0
            
            def traverse(node, parent_id=None):
                nonlocal node_id
                current_id = node_id
                node_type = type(node).__name__
                nodes.append(node_type)
                node_id += 1
                
                if parent_id is not None:
                    edges.append([parent_id, current_id])
                
                for child in ast.iter_child_nodes(node):
                    traverse(child, current_id)
            
            traverse(tree)
            
            if len(nodes) == 0:
                nodes = ['Module']
                edges = []
            
            return nodes, edges
        except:
            return ['Module'], []
    
    def nodes_to_embeddings(self, nodes, tokenizer, device, frozen=False):
        embeddings = []
        for node_type in nodes:
            tokens = tokenizer(node_type, return_tensors='pt', padding=True, truncation=True, max_length=16)
            tokens = {k: v.to(device) for k, v in tokens.items()}
            if frozen:
                with torch.no_grad():
                    outputs = self.graphcodebert(**tokens)
                    embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            else:
                outputs = self.graphcodebert(**tokens)
                embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            embeddings.append(embedding)
        return torch.stack(embeddings)


class CFGBuilder:
    def __init__(self, graphcodebert_model):
        self.graphcodebert = graphcodebert_model
        
    def build_cfg(self, code_snippet):
        try:
            tree = ast.parse(code_snippet)
            nodes = []
            edges = []
            node_id = 0
            
            def traverse(node, prev_id=None):
                nonlocal node_id
                current_id = node_id
                node_type = type(node).__name__
                nodes.append(node_type)
                node_id += 1
                
                if prev_id is not None:
                    edges.append([prev_id, current_id])
                
                if isinstance(node, ast.If):
                    test_id = current_id
                    body_start = node_id
                    for child in node.body:
                        traverse(child, test_id)
                        test_id = node_id - 1
                    
                    orelse_start = node_id
                    for child in node.orelse:
                        traverse(child, current_id)
                    return
                
                elif isinstance(node, (ast.For, ast.While)):
                    loop_id = current_id
                    for child in ast.iter_child_nodes(node):
                        traverse(child, loop_id)
                        loop_id = node_id - 1
                    edges.append([node_id - 1, current_id])
                    return
                
                else:
                    prev = current_id
                    for child in ast.iter_child_nodes(node):
                        traverse(child, prev)
                        prev = node_id - 1
            
            traverse(tree)
            
            if len(nodes) == 0:
                nodes = ['Module']
                edges = []
            
            return nodes, edges
        except:
            return ['Module'], []
    
    def nodes_to_embeddings(self, nodes, tokenizer, device, frozen=False):
        embeddings = []
        for node_type in nodes:
            tokens = tokenizer(node_type, return_tensors='pt', padding=True, truncation=True, max_length=16)
            tokens = {k: v.to(device) for k, v in tokens.items()}
            if frozen:
                with torch.no_grad():
                    outputs = self.graphcodebert(**tokens)
                    embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            else:
                outputs = self.graphcodebert(**tokens)
                embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            embeddings.append(embedding)
        return torch.stack(embeddings)


class StableGraphEncoder(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=384, output_dim=512, num_layers=2):
        super().__init__()
        self.num_layers = num_layers
        
        if num_layers == 1:
            self.conv1 = GCNConv(input_dim, output_dim)
            self.norm1 = nn.LayerNorm(output_dim)
            self.residual = nn.Linear(input_dim, output_dim)
        elif num_layers == 2:
            self.conv1 = GCNConv(input_dim, hidden_dim)
            self.conv2 = GCNConv(hidden_dim, output_dim)
            self.norm1 = nn.LayerNorm(hidden_dim)
            self.norm2 = nn.LayerNorm(output_dim)
            self.residual = nn.Linear(input_dim, output_dim)
        elif num_layers == 3:
            self.conv1 = GCNConv(input_dim, hidden_dim)
            self.conv2 = GCNConv(hidden_dim, hidden_dim)
            self.conv3 = GCNConv(hidden_dim, output_dim)
            self.norm1 = nn.LayerNorm(hidden_dim)
            self.norm2 = nn.LayerNorm(hidden_dim)
            self.norm3 = nn.LayerNorm(output_dim)
            self.residual = nn.Linear(input_dim, output_dim)
        
        self.dropout = nn.Dropout(0.2)
    
    def forward(self, x, edge_index):
        identity = self.residual(x)
        
        if self.num_layers == 1:
            x = self.conv1(x, edge_index)
            x = self.norm1(x)
            x = x + identity
        elif self.num_layers == 2:
            x = self.conv1(x, edge_index)
            x = self.norm1(x)
            x = F.gelu(x)
            x = self.dropout(x)
            x = self.conv2(x, edge_index)
            x = self.norm2(x)
            x = x + identity
        elif self.num_layers == 3:
            x = self.conv1(x, edge_index)
            x = self.norm1(x)
            x = F.gelu(x)
            x = self.dropout(x)
            x = self.conv2(x, edge_index)
            x = self.norm2(x)
            x = F.gelu(x)
            x = self.dropout(x)
            x = self.conv3(x, edge_index)
            x = self.norm3(x)
            x = x + identity
        
        return x


class HierarchicalGraphFusion(nn.Module):
    def __init__(self, graph_dim=512, num_heads=8, num_layers=2, num_graphs=3):
        super().__init__()
        self.graph_dim = graph_dim
        self.num_graphs = num_graphs
        
        self.icfg_encoder = StableGraphEncoder(768, 384, graph_dim, num_layers=num_layers)
        self.dfg_encoder = StableGraphEncoder(768, 384, graph_dim, num_layers=num_layers)
        self.cdg_encoder = StableGraphEncoder(768, 384, graph_dim, num_layers=num_layers)
        
        self.graph_attention = nn.MultiheadAttention(graph_dim, num_heads, dropout=0.1, batch_first=True)
        self.graph_norm = nn.LayerNorm(graph_dim)
        
        self.gate = nn.Sequential(
            nn.Linear(graph_dim * num_graphs, graph_dim),
            nn.Sigmoid()
        )
        
        self.fusion = nn.Sequential(
            nn.Linear(graph_dim * num_graphs, graph_dim * 2),
            nn.LayerNorm(graph_dim * 2),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(graph_dim * 2, graph_dim)
        )
        
    def forward(self, icfg_data, dfg_data, cdg_data):
        graphs_global = []
        
        if icfg_data is not None:
            h_icfg = self.icfg_encoder(icfg_data.x, icfg_data.edge_index)
            icfg_global = torch.mean(h_icfg, dim=0, keepdim=True)
            graphs_global.append(icfg_global)
        
        if dfg_data is not None:
            h_dfg = self.dfg_encoder(dfg_data.x, dfg_data.edge_index)
            dfg_global = torch.mean(h_dfg, dim=0, keepdim=True)
            graphs_global.append(dfg_global)
        
        if cdg_data is not None:
            h_cdg = self.cdg_encoder(cdg_data.x, cdg_data.edge_index)
            cdg_global = torch.mean(h_cdg, dim=0, keepdim=True)
            graphs_global.append(cdg_global)
        
        graph_stack = torch.stack(graphs_global, dim=1)
        
        attended, _ = self.graph_attention(graph_stack, graph_stack, graph_stack)
        attended = self.graph_norm(attended + graph_stack)
        
        fused = torch.cat([attended[:, i] for i in range(len(graphs_global))], dim=-1)
        
        gate_weights = self.gate(fused)
        output = self.fusion(fused)
        output = output * gate_weights
        
        return output.squeeze(0)


class AblationModel(nn.Module):
    def __init__(self, config, num_classes=6):
        super().__init__()
        self.config = config
        self.tokenizer = RobertaTokenizer.from_pretrained('microsoft/graphcodebert-base')
        self.graphcodebert = RobertaModel.from_pretrained('microsoft/graphcodebert-base')
        
        if config.get('finetune_bert', False):
            for param in self.graphcodebert.parameters():
                param.requires_grad = True
        else:
            for param in self.graphcodebert.parameters():
                param.requires_grad = False
        
        self.icfg_builder = ICFGBuilder(self.graphcodebert)
        self.dfg_builder = DFGBuilder(self.graphcodebert)
        self.cdg_builder = CDGBuilder(self.graphcodebert)
        self.ast_builder = ASTBuilder(self.graphcodebert)
        self.cfg_builder = CFGBuilder(self.graphcodebert)
        
        num_graphs = 0
        if config.get('use_icfg', False): num_graphs += 1
        if config.get('use_dfg', False): num_graphs += 1
        if config.get('use_cdg', False): num_graphs += 1
        if config.get('use_ast', False): num_graphs += 1
        if config.get('use_cfg', False): num_graphs += 1
        
        if config.get('use_graphs', False) and num_graphs > 0:
            gcn_layers = config.get('gcn_layers', 2)
            self.graph_fusion = HierarchicalGraphFusion(
                graph_dim=512, 
                num_heads=8, 
                num_layers=gcn_layers,
                num_graphs=num_graphs
            )
        
        self.code_projection = nn.Linear(768, 512)
        
        if config.get('use_graphs', False) and num_graphs > 0:
            if config.get('use_fusion', False):
                self.multimodal_fusion = nn.Sequential(
                    nn.Linear(512 + 512, 768),
                    nn.LayerNorm(768),
                    nn.GELU(),
                    nn.Dropout(0.2),
                    nn.Linear(768, 512),
                    nn.LayerNorm(512)
                )
                classifier_input = 512
            else:
                classifier_input = 512 + 512
        else:
            classifier_input = 512
        
        if not config.get('use_code_text', True):
            if config.get('use_graphs', False) and num_graphs > 0:
                classifier_input = 512
            else:
                classifier_input = 768
        
        self.classifier = nn.Sequential(
            nn.Linear(classifier_input, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
        
        self.criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
        
    def build_graph_data(self, code, device):
        frozen_embeddings = self.config.get('frozen_graph_embeddings', False)
        
        graphs = {}
        
        if self.config.get('use_icfg', False):
            icfg_nodes, icfg_edges = self.icfg_builder.build_icfg(code)
            icfg_x = self.icfg_builder.nodes_to_embeddings(icfg_nodes, self.tokenizer, device, frozen=frozen_embeddings)
            if len(icfg_edges) == 0:
                icfg_edge_index = torch.tensor([[0], [0]], dtype=torch.long, device=device)
            else:
                icfg_edge_index = torch.tensor(icfg_edges, dtype=torch.long, device=device).t().contiguous()
            graphs['icfg'] = Data(x=icfg_x, edge_index=icfg_edge_index)
        
        if self.config.get('use_dfg', False):
            dfg_nodes, dfg_edges = self.dfg_builder.build_dfg(code)
            dfg_x = self.dfg_builder.nodes_to_embeddings(dfg_nodes, self.tokenizer, device, frozen=frozen_embeddings)
            if len(dfg_edges) == 0:
                dfg_edge_index = torch.tensor([[0], [0]], dtype=torch.long, device=device)
            else:
                dfg_edge_index = torch.tensor(dfg_edges, dtype=torch.long, device=device).t().contiguous()
            graphs['dfg'] = Data(x=dfg_x, edge_index=dfg_edge_index)
        
        if self.config.get('use_cdg', False):
            cdg_nodes, cdg_edges = self.cdg_builder.build_cdg(code)
            cdg_x = self.cdg_builder.nodes_to_embeddings(cdg_nodes, self.tokenizer, device, frozen=frozen_embeddings)
            if len(cdg_edges) == 0:
                cdg_edge_index = torch.tensor([[0], [0]], dtype=torch.long, device=device)
            else:
                cdg_edge_index = torch.tensor(cdg_edges, dtype=torch.long, device=device).t().contiguous()
            graphs['cdg'] = Data(x=cdg_x, edge_index=cdg_edge_index)
        
        if self.config.get('use_ast', False):
            ast_nodes, ast_edges = self.ast_builder.build_ast(code)
            ast_x = self.ast_builder.nodes_to_embeddings(ast_nodes, self.tokenizer, device, frozen=frozen_embeddings)
            if len(ast_edges) == 0:
                ast_edge_index = torch.tensor([[0], [0]], dtype=torch.long, device=device)
            else:
                ast_edge_index = torch.tensor(ast_edges, dtype=torch.long, device=device).t().contiguous()
            graphs['ast'] = Data(x=ast_x, edge_index=ast_edge_index)
        
        if self.config.get('use_cfg', False):
            cfg_nodes, cfg_edges = self.cfg_builder.build_cfg(code)
            cfg_x = self.cfg_builder.nodes_to_embeddings(cfg_nodes, self.tokenizer, device, frozen=frozen_embeddings)
            if len(cfg_edges) == 0:
                cfg_edge_index = torch.tensor([[0], [0]], dtype=torch.long, device=device)
            else:
                cfg_edge_index = torch.tensor(cfg_edges, dtype=torch.long, device=device).t().contiguous()
            graphs['cfg'] = Data(x=cfg_x, edge_index=cfg_edge_index)
        
        return graphs
        
    def forward(self, code):
        device = next(self.parameters()).device
        
        representations = []
        
        if self.config.get('use_graphs', False):
            graphs = self.build_graph_data(code, device)
            if len(graphs) > 0:
                icfg_data = graphs.get('icfg', None) if self.config.get('use_icfg', False) else None
                dfg_data = graphs.get('dfg', None) if self.config.get('use_dfg', False) else None
                cdg_data = graphs.get('cdg', None) if self.config.get('use_cdg', False) else None
                
                if self.config.get('use_ast', False):
                    icfg_data = graphs.get('ast', None)
                if self.config.get('use_cfg', False):
                    dfg_data = graphs.get('cfg', None)
                    cdg_data = None
                
                graph_repr = self.graph_fusion(icfg_data, dfg_data, cdg_data)
                representations.append(graph_repr.unsqueeze(0))
        
        if self.config.get('use_code_text', True):
            tokens = self.tokenizer(
                code, 
                return_tensors='pt', 
                truncation=True, 
                max_length=512, 
                padding='max_length'
            )
            tokens = {k: v.to(device) for k, v in tokens.items()}
            
            code_output = self.graphcodebert(**tokens)
            code_repr = code_output.last_hidden_state[:, 0, :]
            code_repr = self.code_projection(code_repr)
            representations.append(code_repr)
        
        if len(representations) == 0:
            tokens = self.tokenizer(
                code, 
                return_tensors='pt', 
                truncation=True, 
                max_length=512, 
                padding='max_length'
            )
            tokens = {k: v.to(device) for k, v in tokens.items()}
            code_output = self.graphcodebert(**tokens)
            fused_repr = code_output.last_hidden_state[:, 0, :]
        elif len(representations) == 1:
            fused_repr = representations[0]
        else:
            if self.config.get('use_fusion', False):
                combined = torch.cat(representations, dim=-1)
                fused_repr = self.multimodal_fusion(combined)
            else:
                fused_repr = torch.cat(representations, dim=-1)
        
        logits = self.classifier(fused_repr)
        return logits


class CodeDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return {
            'func': str(row['func']),
            'label': int(row['label'])
        }


def train_model(model, train_loader, val_loader, num_epochs=20, device='mps', model_name='model'):
    optimizer = torch.optim.AdamW([
        {'params': model.graphcodebert.parameters(), 'lr': 5e-6, 'weight_decay': 0.01},
        {'params': [p for n, p in model.named_parameters() if 'graphcodebert' not in n], 'lr': 1e-4, 'weight_decay': 0.01}
    ])
    
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=3, T_mult=2, eta_min=1e-7
    )
    
    best_val_f1 = 0
    patience = 8
    patience_counter = 0
    
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0
        train_preds = []
        train_labels = []
        
        progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')
        for batch in progress_bar:
            code = batch['func'][0] if isinstance(batch['func'], list) else batch['func']
            label = batch['label']
            
            if isinstance(label, torch.Tensor):
                if label.dim() == 0:
                    label = label.item()
                else:
                    label = label[0].item() if len(label) > 0 else label.item()
            
            optimizer.zero_grad()
            
            logits = model(code)
            
            if logits.dim() == 1:
                logits = logits.unsqueeze(0)
            
            label_tensor = torch.tensor([label], dtype=torch.long, device=device)
            loss = model.criterion(logits, label_tensor)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            optimizer.step()
            
            train_loss += loss.item()
            pred = torch.argmax(logits, dim=1).cpu().item()
            train_preds.append(pred)
            train_labels.append(label)
            
            progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        scheduler.step()
        
        train_acc = accuracy_score(train_labels, train_preds)
        _, _, train_f1, _ = precision_recall_fscore_support(
            train_labels, train_preds, average='weighted', zero_division=0
        )
        
        model.eval()
        val_preds = []
        val_labels = []
        
        with torch.no_grad():
            for batch in val_loader:
                code = batch['func'][0] if isinstance(batch['func'], list) else batch['func']
                label = batch['label']
                
                if isinstance(label, torch.Tensor):
                    if label.dim() == 0:
                        label = label.item()

                    else:
                        label = label[0].item() if len(label) > 0 else label.item()
                
                logits = model(code)
                
                if logits.dim() == 1:
                    logits = logits.unsqueeze(0)
                
                pred = torch.argmax(logits, dim=1).cpu().item()
                val_preds.append(pred)
                val_labels.append(label)
        
        val_acc = accuracy_score(val_labels, val_preds)
        _, _, val_f1, _ = precision_recall_fscore_support(
            val_labels, val_preds, average='weighted', zero_division=0
        )
        
        print(f'\nEpoch {epoch+1}/{num_epochs}:')
        print(f'  Train: Loss={train_loss/len(train_loader):.4f}, Acc={train_acc:.4f}, F1={train_f1:.4f}')
        print(f'  Val:   Acc={val_acc:.4f}, F1={val_f1:.4f}')
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            patience_counter = 0
            torch.save(model.state_dict(), f'best_{model_name}.pt')
            print(f'  ✓ Best model saved (F1: {best_val_f1:.4f})')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'\nEarly stopping at epoch {epoch+1}')
                break
    
    print(f'\nLoading best model (F1: {best_val_f1:.4f})')
    model.load_state_dict(torch.load(f'best_{model_name}.pt', map_location=device, weights_only=True))
    return model


def evaluate_model(model, test_loader, device='mps'):
    model.eval()
    test_preds = []
    test_labels = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc='Testing'):
            code = batch['func'][0] if isinstance(batch['func'], list) else batch['func']
            label = batch['label']
            
            if isinstance(label, torch.Tensor):
                if label.dim() == 0:
                    label = label.item()
                else:
                    label = label[0].item() if len(label) > 0 else label.item()
            
            logits = model(code)
            
            if logits.dim() == 1:
                logits = logits.unsqueeze(0)
            
            pred = torch.argmax(logits, dim=1).cpu().item()
            test_preds.append(pred)
            test_labels.append(label)
    
    test_acc = accuracy_score(test_labels, test_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        test_labels, test_preds, average='weighted', zero_division=0
    )
    
    return test_acc, precision, recall, f1


if __name__ == '__main__':
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    print(f'Device: {device}')

    df = pd.read_csv('/Users/akter/fahim/data/trainpro (1).csv')
    train_label0_sample = df[df['label'] == 0].sample(n=3800, random_state=42)
    train_others = df[df['label'] != 0]
    df = pd.concat([train_label0_sample, train_others], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)
    
    print(f'\n{"="*70}')
    print('Dataset Information:')
    print(f'  Shape: {df.shape}')
    print(f'  Columns: {df.columns.tolist()}')
    print('\nLabel Distribution:')
    print(df["label"].value_counts().sort_index())
    
    train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['label'])
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])
    
    print(f'\nData Split:')
    print(f'  Train: {len(train_df)}')
    print(f'  Val:   {len(val_df)}')
    print(f'  Test:  {len(test_df)}')
    
    train_dataset = CodeDataset(train_df)
    val_dataset = CodeDataset(val_df)
    test_dataset = CodeDataset(test_df)
    
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=8, shuffle=True)
    val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=8, shuffle=False)
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=8, shuffle=False)
    
    configs = [
        # {
        #     'name': 'GraphCodeBERT',
        #     'use_graphs': False,
        #     'use_fusion': False,
        #     'finetune_bert': False,
        #     'use_code_text': True,
        #     'use_icfg': False,
        #     'use_dfg': False,
        #     'use_cdg': False,
        #     'use_ast': False,
        #     'use_cfg': False,
        #     'gcn_layers': 2,
        #     'frozen_graph_embeddings': False
        # },
        # {
        #     'name': 'GCB+Finetune',
        #     'use_graphs': False,
        #     'use_fusion': False,
        #     'finetune_bert': True,
        #     'use_code_text': True,
        #     'use_icfg': False,
        #     'use_dfg': False,
        #     'use_cdg': False,
        #     'use_ast': False,
        #     'use_cfg': False,
        #     'gcn_layers': 2,
        #     'frozen_graph_embeddings': False
        # },
        # {
        #     'name': 'GCB+Graphs',
        #     'use_graphs': True,
        #     'use_fusion': False,
        #     'finetune_bert': False,
        #     'use_code_text': True,
        #     'use_icfg': True,
        #     'use_dfg': True,
        #     'use_cdg': True,
        #     'use_ast': False,
        #     'use_cfg': False,
        #     'gcn_layers': 2,
        #     'frozen_graph_embeddings': False
        # },
        # {
        #     'name': 'GCB+Graphs+FT',
        #     'use_graphs': True,
        #     'use_fusion': False,
        #     'finetune_bert': True,
        #     'use_code_text': True,
        #     'use_icfg': True,
        #     'use_dfg': True,
        #     'use_cdg': True,
        #     'use_ast': False,
        #     'use_cfg': False,
        #     'gcn_layers': 2,
        #     'frozen_graph_embeddings': False
        # },
        # {
        #     'name': 'GCB+Fusion',
        #     'use_graphs': False,
        #     'use_fusion': True,
        #     'finetune_bert': False,
        #     'use_code_text': True,
        #     'use_icfg': False,
        #     'use_dfg': False,
        #     'use_cdg': False,
        #     'use_ast': False,
        #     'use_cfg': False,
        #     'gcn_layers': 2,
        #     'frozen_graph_embeddings': False
        # },
        {
            'name': 'Full(Proposed)',
            'use_graphs': True,
            'use_fusion': True,
            'finetune_bert': True,
            'use_code_text': True,
            'use_icfg': True,
            'use_dfg': True,
            'use_cdg': True,
            'use_ast': False,
            'use_cfg': False,
            'gcn_layers': 2,
            'frozen_graph_embeddings': False
        },
        {
            'name': 'Full-ICFG',
            'use_graphs': True,
            'use_fusion': True,
            'finetune_bert': True,
            'use_code_text': True,
            'use_icfg': False,
            'use_dfg': True,
            'use_cdg': True,
            'use_ast': False,
            'use_cfg': False,
            'gcn_layers': 2,
            'frozen_graph_embeddings': False
        },
        {
            'name': 'Full-DFG',
            'use_graphs': True,
            'use_fusion': True,
            'finetune_bert': True,
            'use_code_text': True,
            'use_icfg': True,
            'use_dfg': False,
            'use_cdg': True,
            'use_ast': False,
            'use_cfg': False,
            'gcn_layers': 2,
            'frozen_graph_embeddings': False
        },
        {
            'name': 'Full-CDG',
            'use_graphs': True,
            'use_fusion': True,
            'finetune_bert': True,
            'use_code_text': True,
            'use_icfg': True,
            'use_dfg': True,
            'use_cdg': False,
            'use_ast': False,
            'use_cfg': False,
            'gcn_layers': 2,
            'frozen_graph_embeddings': False
        },
        {
            'name': 'ICFG+DFG',
            'use_graphs': True,
            'use_fusion': True,
            'finetune_bert': True,
            'use_code_text': True,
            'use_icfg': True,
            'use_dfg': True,
            'use_cdg': False,
            'use_ast': False,
            'use_cfg': False,
            'gcn_layers': 2,
            'frozen_graph_embeddings': False
        },
        {
            'name': 'ICFG+CDG',
            'use_graphs': True,
            'use_fusion': True,
            'finetune_bert': True,
            'use_code_text': True,
            'use_icfg': True,
            'use_dfg': False,
            'use_cdg': True,
            'use_ast': False,
            'use_cfg': False,
            'gcn_layers': 2,
            'frozen_graph_embeddings': False
        },
        {
            'name': 'DFG+CDG',
            'use_graphs': True,
            'use_fusion': True,
            'finetune_bert': True,
            'use_code_text': True,
            'use_icfg': False,
            'use_dfg': True,
            'use_cdg': True,
            'use_ast': False,
            'use_cfg': False,
            'gcn_layers': 2,
            'frozen_graph_embeddings': False
        },
        {
            'name': 'Full+AST',
            'use_graphs': True,
            'use_fusion': True,
            'finetune_bert': True,
            'use_code_text': True,
            'use_icfg': False,
            'use_dfg': False,
            'use_cdg': False,
            'use_ast': True,
            'use_cfg': False,
            'gcn_layers': 2,
            'frozen_graph_embeddings': False
        },
        {
            'name': 'Full+CFG',
            'use_graphs': True,
            'use_fusion': True,
            'finetune_bert': True,
            'use_code_text': True,
            'use_icfg': False,
            'use_dfg': False,
            'use_cdg': False,
            'use_ast': False,
            'use_cfg': True,
            'gcn_layers': 2,
            'frozen_graph_embeddings': False
        },
        {
            'name': 'Full+1GCN',
            'use_graphs': True,
            'use_fusion': True,
            'finetune_bert': True,
            'use_code_text': True,
            'use_icfg': True,
            'use_dfg': True,
            'use_cdg': True,
            'use_ast': False,
            'use_cfg': False,
            'gcn_layers': 1,
            'frozen_graph_embeddings': False
        },
        {
            'name': 'Full+3GCN',
            'use_graphs': True,
            'use_fusion': True,
            'finetune_bert': True,
            'use_code_text': True,
            'use_icfg': True,
            'use_dfg': True,
            'use_cdg': True,
            'use_ast': False,
            'use_cfg': False,
            'gcn_layers': 3,
            'frozen_graph_embeddings': False
        },
        {
            'name': 'Full-CodeText',
            'use_graphs': True,
            'use_fusion': True,
            'finetune_bert': True,
            'use_code_text': False,
            'use_icfg': True,
            'use_dfg': True,
            'use_cdg': True,
            'use_ast': False,
            'use_cfg': False,
            'gcn_layers': 2,
            'frozen_graph_embeddings': False
        },
        {
            'name': 'Full+FrozenEmbed',
            'use_graphs': True,
            'use_fusion': True,
            'finetune_bert': True,
            'use_code_text': True,
            'use_icfg': True,
            'use_dfg': True,
            'use_cdg': True,
            'use_ast': False,
            'use_cfg': False,
            'gcn_layers': 2,
            'frozen_graph_embeddings': True
        }
    ]
    
    results = []
    
    for config in configs:
        print(f'\n{"="*70}')
        print(f'CONFIGURATION: {config["name"]}')
        print(f'{"="*70}')
        print(f'Config: {config}')
        
        model = AblationModel(config, num_classes=6).to(device)
        print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
        
        model = train_model(model, train_loader, val_loader, num_epochs=5, device=device, model_name=config['name'])
        
        test_acc, precision, recall, f1 = evaluate_model(model, test_loader, device=device)
        
        results.append({
            'Configuration': config['name'],
            'Accuracy': test_acc,
            'Precision': precision,
            'Recall': recall,
            'F1-Score': f1
        })
        
        print(f'\n{config["name"]} Results:')
        print(f'  Accuracy:  {test_acc:.4f}')
        print(f'  Precision: {precision:.4f}')
        print(f'  Recall:    {recall:.4f}')
        print(f'  F1 Score:  {f1:.4f}')
        
        torch.save(model.state_dict(), f'final_{config["name"]}.pt')
    
    results_df = pd.DataFrame(results)
    results_df.to_csv('ablation_results.csv', index=False)
    
    print(f'\n{"="*70}')
    print('ABLATION STUDY SUMMARY')
    print(f'{"="*70}')
    print(results_df.to_string(index=False))
    print(f'\nResults saved to: ablation_results.csv')